In [1]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [2]:
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 108.3 MB/s eta 0:00:00


In [3]:
import gc
import tensorflow as tf

tf.keras.backend.clear_session()
gc.collect()

0

In [7]:
import os
import csv
import numpy as np
import tensorflow as tf
from Bio import SeqIO
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from collections import defaultdict
import json
from tensorflow.keras.utils import plot_model
import random
import gc


DATASETS_ROOT = "/content/drive/MyDrive/seq_similarity_8-7/datasets_1-12"
NUM_RUNS = 1

CONFIG = {
    "learning_rate": 5e-4,
    "batch_size": 32,
    "filters_identity": 64,
    "residual_scale": 0.05,
    "mask_loss_pos_weight": 10.0,
    "mask_loss_tv_weight": 0.05,
    "mask_loss_sparsity": 0.002,
    "dense_units_hsp": (128, 64),
}


def set_seed(seed):
    np.random.seed(seed)
    random.seed(seed)
    tf.random.set_seed(seed)


def sample_config(search_space):
    return {
        key: random.choice(values)
        for key, values in search_space.items()
    }


def generate_configs(search_space, num_configs=20):
    keys = list(search_space.keys())

    anchors = {}
    for k, values in search_space.items():
        values = list(values)

        if len(values) <= 2:
            anchors[k] = values
        else:
            anchors[k] = [
                values[0],
                values[len(values)//2],
                values[-1]
            ]

    anchor_configs = []
    for i in range(num_configs // 2):
        cfg = {}
        for k in keys:
            cfg[k] = random.choice(anchors[k])
        anchor_configs.append(cfg)

    random_configs = []
    for i in range(num_configs - len(anchor_configs)):
        cfg = {}
        for k in keys:
            cfg[k] = random.choice(search_space[k])
        random_configs.append(cfg)

    configs = anchor_configs + random_configs

    unique_configs = []
    seen = set()

    for cfg in configs:
        key = tuple(sorted(cfg.items()))
        if key not in seen:
            seen.add(key)
            unique_configs.append(cfg)

    while len(unique_configs) < num_configs:
        cfg = {k: random.choice(search_space[k]) for k in keys}
        key = tuple(sorted(cfg.items()))
        if key not in seen:
            seen.add(key)
            unique_configs.append(cfg)

    return unique_configs[:num_configs]



def create_experiment_dirs(root, num_experiments=20, num_runs=1):
    all_dirs = []

    for exp_id in range(num_experiments):
        exp_dir = os.path.join(root, f"exp_{exp_id}")
        os.makedirs(exp_dir, exist_ok=True)

        run_dirs = []
        for r in range(num_runs):
            rd = os.path.join(exp_dir, f"run_{r}")
            os.makedirs(rd, exist_ok=True)
            run_dirs.append(rd)

        all_dirs.append((exp_dir, run_dirs))

    return all_dirs


def one_hot_encode(seq, max_len):
    mapping = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    arr = np.zeros((max_len, 4), dtype=np.float32)
    for i, base in enumerate(seq[:max_len]):
        if base in mapping:
            arr[i, mapping[base]] = 1.0
    return arr


def get_max_sequence_length(fasta_file):
    return max(len(str(rec.seq)) for rec in SeqIO.parse(fasta_file, "fasta"))


def compute_y_min_max(pairs_file):
    mins = defaultdict(lambda: float("inf"))
    maxs = defaultdict(lambda: float("-inf"))

    with open(pairs_file) as f:
        reader = csv.DictReader(f, delimiter="\t")
        for row in reader:
            for key in ["hsp_identity", "global_identity"]:
                val_str = row[key].strip()
                if not val_str:
                    continue
                val = float(val_str)
                mins[key] = min(mins[key], val)
                maxs[key] = max(maxs[key], val)

    return {
        "hsp_identity": (mins["hsp_identity"], maxs["hsp_identity"]),
        "global_identity": (mins["global_identity"], maxs["global_identity"]),
    }


def load_data(fasta_file, pairs_file, max_len, y_min_max):
    sequences = {r.id: str(r.seq) for r in SeqIO.parse(fasta_file, "fasta")}
    hmin, hmax = y_min_max["hsp_identity"]
    gmin, gmax = y_min_max["global_identity"]

    data = []

    with open(pairs_file) as f:
        reader = csv.DictReader(f, delimiter="\t")
        for row in tqdm(reader):

            qid, sid = row["seq_id_1"], row["seq_id_2"]
            if qid not in sequences or sid not in sequences:
                continue

            if not row["hsp_identity"].strip():
                continue

            hsp_val = float(row["hsp_identity"])
            glob_val = float(row["global_identity"])

            hsp_identity = np.float32((hsp_val - hmin) / (hmax - hmin))
            global_identity = np.float32((glob_val - gmin) / (gmax - gmin))

            q = one_hot_encode(sequences[qid], max_len)
            s = one_hot_encode(sequences[sid], max_len)

            qmask = np.zeros(max_len, dtype=np.float32)
            smask = np.zeros(max_len, dtype=np.float32)

            qstart, qend = int(row["qstart"]), int(row["qend"])
            sstart, send = int(row["sstart"]), int(row["send"])

            qmask[qstart:qend + 1] = 1.0
            smask[sstart:send + 1] = 1.0

            data.append(
                (q, s, qmask, smask, hsp_identity, global_identity)
            )

    return data


def build_dataset(pairs, batch_size=32, shuffle=True):
    q, s, qm, sm, hsp_id, glob_id = zip(*pairs)

    ds = tf.data.Dataset.from_tensor_slices(
        (
            {"q_seq": np.array(q), "s_seq": np.array(s)},
            {
                "qmask": np.array(qm),
                "smask": np.array(sm),
                "hsp_identity": np.array(hsp_id),
                "global_identity": np.array(glob_id),
            }
        )
    )

    if shuffle:
        ds = ds.shuffle(len(pairs), reshuffle_each_iteration=True)

    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)


class AbsDiff(tf.keras.layers.Layer):
    def call(self, inputs):
        a, b = inputs
        return tf.abs(a - b)


class OutputLayer(tf.keras.layers.Layer):
    def __init__(self, name=None, **kwargs):
        super().__init__(name=name, **kwargs)

    def call(self, inputs):
        return inputs


class MaskCoverage(tf.keras.layers.Layer):
    def call(self, mask):
        return tf.reduce_mean(mask, axis=1)


class UniformMask(tf.keras.layers.Layer):
    def __init__(self, seq_len, **kwargs):
        super().__init__(**kwargs)
        self.seq_len = seq_len

    def call(self, x):
        batch_size = tf.shape(x)[0]
        return tf.ones((batch_size, self.seq_len, 1), dtype=tf.float32) / self.seq_len


class MaskedFeatures(tf.keras.layers.Layer):
    def call(self, inputs):
        features, mask = inputs

        if len(mask.shape) == 2:
            mask = tf.expand_dims(mask, -1)

        return features * mask


class CrossAttention(tf.keras.layers.Layer):
    def call(self, inputs):
        q_enc, s_enc = inputs
        scores = tf.matmul(q_enc, s_enc, transpose_b=True)
        probs = tf.nn.softmax(scores, axis=-1)
        return tf.matmul(probs, s_enc)


class MaskWeightedPooling(tf.keras.layers.Layer):
    def call(self, inputs):
        features, mask = inputs
        mask = tf.cast(mask, tf.float32)
        if len(mask.shape) == 2:
            mask = tf.expand_dims(mask, -1)

        numerator = tf.reduce_sum(features * mask, axis=1)
        denominator = tf.reduce_sum(mask, axis=1) + 1e-6
        return numerator / denominator


class ZeroLike(tf.keras.layers.Layer):
    def call(self, x):
        return tf.zeros_like(x)


class IdentityProjection(tf.keras.layers.Layer):
    def __init__(self, feature_dim, **kwargs):
        super().__init__(**kwargs)
        self.feature_dim = feature_dim

    def call(self, identity):
        return tf.repeat(identity, repeats=self.feature_dim, axis=-1)

    def get_config(self):
        config = super().get_config()
        config.update({"feature_dim": self.feature_dim})
        return config


class ResidualScaling(tf.keras.layers.Layer):
    def __init__(self, scale=0.1, **kwargs):
        super().__init__(**kwargs)
        self.scale = scale

    def call(self, x):
        return self.scale * x

    def get_config(self):
        config = super().get_config()
        config.update({"scale": self.scale})
        return config


class MaskedAverage(tf.keras.layers.Layer):
    def call(self, inputs):
        features, mask = inputs

        if len(mask.shape) == 2:
            mask = tf.expand_dims(mask, -1)

        numerator = tf.reduce_sum(features * mask, axis=1)
        denominator = tf.reduce_sum(mask, axis=1) + 1e-6

        return numerator / denominator


class StructuralModelF(tf.keras.layers.Layer):
    def build(self, input_shape):
        self.b1 = self.add_weight(shape=(1,), initializer="ones")
        self.b2 = self.add_weight(shape=(1,), initializer="zeros")
        self.b3 = self.add_weight(shape=(1,), initializer="zeros")
        self.b4 = self.add_weight(shape=(1,), initializer="zeros")
        self.b5 = self.add_weight(shape=(1,), initializer="zeros")
        self.bias = self.add_weight(shape=(1,), initializer="zeros")

    def call(self, inputs):
        hsp_identity, coverage = inputs

        return (
                self.b1 * hsp_identity +
                self.b2 * coverage +
                self.b3 * (hsp_identity * coverage) +
                self.b4 * tf.square(hsp_identity) +
                self.b5 * tf.square(coverage) +
                self.bias
        )


def build_positional_encoder(seq_len, vocab_size=4, filters=32, depth=5, use_residuals=True, use_dilation=True):
    inp = tf.keras.Input(shape=(seq_len, vocab_size))
    x = tf.keras.layers.Conv1D(filters, 7, padding="same", activation="relu")(inp)

    for i in range(depth):
        res = x
        dilation = 2 ** i if use_dilation else 1
        x = tf.keras.layers.SeparableConv1D(filters, 3, padding="same", dilation_rate=dilation, activation="relu")(x)
        x = tf.keras.layers.BatchNormalization()(x)

        if use_residuals:
            x = tf.keras.layers.Add()([x, res])

    pooled = tf.keras.layers.GlobalMaxPooling1D()(x)
    return tf.keras.Model(inp, [x, pooled], name="positional_encoder")


def build_identity_encoder(seq_len, vocab_size=4, filters=64):
    inp = tf.keras.Input(shape=(seq_len, vocab_size))
    x = tf.keras.layers.Conv1D(filters, 7, padding="same", activation="relu")(inp)
    x = tf.keras.layers.Conv1D(filters, 5, padding="same", activation="relu")(x)
    return tf.keras.Model(inp, x, name="identity_encoder")


def build_positional_decoder(seq_len, channels, hidden_1=64, hidden_2=32):
    inp = tf.keras.Input((seq_len, channels))
    x = tf.keras.layers.Conv1D(hidden_1, 3, padding="same", activation="relu")(inp)
    x = tf.keras.layers.Conv1D(hidden_2, 3, padding="same", activation="relu")(x)
    mask = tf.keras.layers.Conv1D(1, 1, activation="sigmoid", dtype="float32", name="mask")(x)
    return tf.keras.Model(inp, mask, name="mask_decoder")



def build_model_ablation(seq_len, config, vocab_size=4):
    pos_filters = config.get("pos_filters", 32)
    pos_depth = config.get("pos_depth", 5)
    identity_filters = config.get("filters_identity", 64)

    dense_units = config.get("dense_units_hsp", (128, 64))
    residual_scale = config.get("residual_scale", 0.05)

    decoder_hidden = config.get("decoder_hidden", (64, 32))

    q_in = tf.keras.Input((seq_len, vocab_size), name="q_seq")
    s_in = tf.keras.Input((seq_len, vocab_size), name="s_seq")

    positional_encoder = build_positional_encoder(
        seq_len,
        vocab_size=vocab_size,
        filters=pos_filters,
        depth=pos_depth,
        use_residuals=True,
        use_dilation=True
    )

    q_seq, _ = positional_encoder(q_in)
    s_seq, _ = positional_encoder(s_in)

    attn_qs = CrossAttention()([q_seq, s_seq])
    attn_sq = CrossAttention()([s_seq, q_seq])

    decoder = build_positional_decoder(seq_len, q_seq.shape[-1] * 4, hidden_1=decoder_hidden[0], hidden_2=decoder_hidden[1])

    def decode(seq, attn):
        x = tf.keras.layers.Concatenate()([
            seq,
            attn,
            AbsDiff()([seq, attn]),
            tf.keras.layers.Multiply()([seq, attn])
        ])
        return decoder(x)

    qmask = decode(q_seq, attn_qs)
    smask = decode(s_seq, attn_sq)


    qmask = OutputLayer(name="qmask")(qmask)
    smask = OutputLayer(name="smask")(smask)

    identity_encoder_model = build_identity_encoder(seq_len, vocab_size, filters=identity_filters)

    q_features = identity_encoder_model(q_in)
    s_features = identity_encoder_model(s_in)

    q_features = MaskedFeatures()([q_features, qmask])
    s_features = MaskedFeatures()([s_features, smask])

    q_hsp = MaskWeightedPooling()([q_features, qmask])
    s_hsp = MaskWeightedPooling()([s_features, smask])

    x_hsp = AbsDiff()([q_hsp, s_hsp])
    x_hsp = tf.keras.layers.Dense(dense_units[0], activation="relu")(x_hsp)
    x_hsp = tf.keras.layers.Dense(dense_units[1], activation="relu")(x_hsp)

    hsp_identity = tf.keras.layers.Dense(1, name="hsp_identity")(x_hsp)

    q_cov = MaskCoverage()(qmask)
    s_cov = MaskCoverage()(smask)
    mean_cov = (q_cov + s_cov) / 2.0

    base = StructuralModelF()([hsp_identity, mean_cov])

    residual = tf.keras.layers.Dense(32, activation="relu")(x_hsp)
    residual = tf.keras.layers.Dense(1)(residual)
    residual = ResidualScaling(scale=residual_scale)(residual)

    global_identity = tf.keras.layers.Add(name="global_identity")([base, residual])

    return tf.keras.Model(
        inputs=[q_in, s_in],
        outputs={
            "qmask": qmask,
            "smask": smask,
            "hsp_identity": hsp_identity,
            "global_identity": global_identity
        }
    )


def hsp_mask_loss(pos_weight=10.0, tv_weight=0.05, sparsity_weight=0.002):
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)

        bce = tf.keras.backend.binary_crossentropy(y_true, y_pred)
        weights = 1.0 + y_true * (pos_weight - 1.0)
        bce = tf.reduce_mean(bce * weights)

        tv = tf.reduce_mean(tf.square(y_pred[:, 1:] - y_pred[:, :-1]))

        sparsity = tf.reduce_mean(y_pred)

        return bce + tv_weight * tv + sparsity_weight * sparsity

    return loss


def mask_iou(y_true, y_pred):
    if len(y_pred.shape) == 3:
        y_pred = tf.squeeze(y_pred, axis=-1)

    y_pred_bin = tf.cast(y_pred > 0.5, tf.float32)

    intersection = tf.reduce_sum(y_true * y_pred_bin, axis=1)
    union = tf.reduce_sum(tf.maximum(y_true, y_pred_bin), axis=1)

    iou = intersection / (union + 1e-7)
    return tf.reduce_mean(iou)


def save_history_plots(history, out_dir="train_history"):
    os.makedirs(out_dir, exist_ok=True)

    for key, values in history.history.items():
        if key.startswith("val_"):
            dataset = "val"
            name = key[4:]
            color = "orange"
        else:
            dataset = "train"
            name = key
            color = "#450558"

        plt.figure(figsize=(3, 3), dpi=200)
        plt.plot(range(1, len(values) + 1), values, linewidth=1.2, color=color)
        plt.title(f"{dataset.capitalize()} {name.replace('_', ' ').title()}", fontsize=10)
        plt.xlabel("Epoch", fontsize=10)
        plt.ylabel(name.replace("_", " ").title(), fontsize=10)
        plt.xticks(fontsize=10)
        plt.yticks(fontsize=10)
        plt.grid(True, linewidth=0.4)
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, f"{dataset}_{name}.png"), bbox_inches="tight", pad_inches=0.05)
        plt.close()


dataset_dirs = sorted([
    os.path.join(DATASETS_ROOT, d)
    for d in os.listdir(DATASETS_ROOT)
    if d.startswith("dataset_")
])

for dataset_dir in dataset_dirs:

    print(f"\n=== DATASET: {dataset_dir} ===")

    fasta_path = os.path.join(dataset_dir, "train_sequences.fasta")
    tsv_path = os.path.join(dataset_dir, "train_labels.tsv")
    json_path = os.path.join(dataset_dir, "metadata.json")

    out_root = os.path.join(dataset_dir, "fixed_model_runs")
    os.makedirs(out_root, exist_ok=True)

    MAX_SEQ_LEN = get_max_sequence_length(fasta_path)
    y_min_max = compute_y_min_max(tsv_path)

    metadata = {
        "max_sequence_length": int(MAX_SEQ_LEN),
        "hsp_identity": {
            "min": float(y_min_max["hsp_identity"][0]),
            "max": float(y_min_max["hsp_identity"][1]),
        },
        "global_identity": {
            "min": float(y_min_max["global_identity"][0]),
            "max": float(y_min_max["global_identity"][1]),
        }
    }

    with open(json_path, "w") as f:
        json.dump(metadata, f, indent=2)

    pairs = load_data(fasta_path, tsv_path, MAX_SEQ_LEN, y_min_max)

    idx = np.arange(len(pairs))
    train_idx, val_idx = train_test_split(idx, test_size=0.3, random_state=42)

    train_pairs = [pairs[i] for i in train_idx]
    val_pairs = [pairs[i] for i in val_idx]


    for run_idx in range(NUM_RUNS):

        print(f"\n--- Run {run_idx} ---")

        run_dir = os.path.join(out_root, f"run_{run_idx}")
        os.makedirs(run_dir, exist_ok=True)

        tf.keras.backend.clear_session()
        gc.collect()
        set_seed(42 + run_idx)

        train_ds = build_dataset(
            train_pairs,
            batch_size=CONFIG["batch_size"],
            shuffle=True
        )

        val_ds = build_dataset(
            val_pairs,
            batch_size=CONFIG["batch_size"],
            shuffle=False
        )

        loss = {
            "hsp_identity": tf.keras.losses.Huber(delta=0.1),
            "global_identity": tf.keras.losses.Huber(delta=0.1),
        }

        metrics = {
            "hsp_identity": tf.keras.metrics.MeanAbsoluteError(),
            "global_identity": tf.keras.metrics.MeanAbsoluteError(),
        }

        loss["qmask"] = hsp_mask_loss(
            CONFIG["mask_loss_pos_weight"],
            CONFIG["mask_loss_tv_weight"],
            CONFIG["mask_loss_sparsity"]
        )
        loss["smask"] = hsp_mask_loss(
            CONFIG["mask_loss_pos_weight"],
            CONFIG["mask_loss_tv_weight"],
            CONFIG["mask_loss_sparsity"]
        )

        model = build_model_ablation(MAX_SEQ_LEN, CONFIG)

        model_info_path = os.path.join(out_root, "model_info.json")

        if not os.path.exists(model_info_path):
            total_params = int(model.count_params())

            model_info = {
                "total_params": total_params,
                "config": CONFIG
            }

            with open(model_info_path, "w") as f:
                json.dump(model_info, f, indent=2)

            print(f"Saved model params: {total_params}")



        model.compile(
            optimizer=tf.keras.optimizers.Adam(CONFIG["learning_rate"]),
            loss=loss,
            metrics=metrics
        )

        callbacks = [
            tf.keras.callbacks.EarlyStopping(
                monitor="val_global_identity_mean_absolute_error",
                mode="min",
                patience=2,
                restore_best_weights=True
            )
        ]

        model.summary()
        plot_model(model, to_file=os.path.join(run_dir, "architecture.png"), show_shapes=True, show_layer_names=True, rankdir="TB", expand_nested=True)
        history = model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=15,
            callbacks=callbacks,
            verbose=1
        )

        val_mae = history.history["val_global_identity_mean_absolute_error"]
        val_hsp = history.history["val_hsp_identity_mean_absolute_error"]

        best_epoch = int(np.argmin(val_mae))
        best_val_global_mae = float(val_mae[best_epoch])
        best_val_hsp_mae = float(val_hsp[best_epoch])

        results = {
            "best_epoch": best_epoch + 1,
            "best_val_global_mae": best_val_global_mae,
            "best_val_hsp_mae": best_val_hsp_mae,
            "final_epoch": len(val_mae),
            "seed": 42 + run_idx
        }

        with open(os.path.join(run_dir, "metrics.json"), "w") as f:
            json.dump(results, f, indent=2)

        print("Saved metrics:", results)

        model.save(os.path.join(run_dir, "model.keras"))
        save_history_plots(history, out_dir=os.path.join(run_dir, "train_history"))



=== DATASET: /content/drive/MyDrive/seq_similarity_8-7/datasets_1-12/dataset_10 ===


11897it [00:07, 1585.35it/s]



--- Run 0 ---
Saved model params: 79337


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ q_seq (InputLayer)  │ (None, 1692, 4)   │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ s_seq (InputLayer)  │ (None, 1692, 4)   │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_encoder  │ [(None, 1692,     │      7,328 │ q_seq[0][0],      │
│ (Functional)        │ 32), (None, 32)]  │            │ s_seq[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cross_attention     │ (None, 1692, 32)  │          0 │ positional_encod… │
│ (CrossAttention)    │                   │            │ positional_encod… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cross_attention_1   │ (None, 1692, 32)  │          0 │ positional_encod… │
│ (CrossAttention)    │                   │            │ positional_encod… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ abs_diff (AbsDiff)  │ (None, 1692, 32)  │          0 │ positional_encod… │
│                     │                   │            │ cross_attention[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 1692, 32)  │          0 │ positional_encod… │
│                     │                   │            │ cross_attention[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ abs_diff_1          │ (None, 1692, 32)  │          0 │ positional_encod… │
│ (AbsDiff)           │                   │            │ cross_attention_… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_1          │ (None, 1692, 32)  │          0 │ positional_encod… │
│ (Multiply)          │                   │            │ cross_attention_… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 1692, 128) │          0 │ positional_encod… │
│ (Concatenate)       │                   │            │ cross_attention[… │
│                     │                   │            │ abs_diff[0][0],   │
│                     │                   │            │ multiply[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 1692, 128) │          0 │ positional_encod… │
│ (Concatenate)       │                   │            │ cross_attention_… │
│                     │                   │            │ abs_diff_1[0][0], │
│                     │                   │            │ multiply_1[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mask_decoder        │ (None, 1692, 1)   │     30,849 │ concatenate[0][0… │
│ (Functional)        │                   │            │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ identity_encoder    │ (None, 1692, 64)  │     22,400 │ q_seq[0][0],      │
│ (Functional)        │                   │            │ s_seq[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ qmask (OutputLayer) │ (None, 1692, 1)   │          0 │ mask_decoder[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ smask (OutputLayer) │ (None, 1692, 1)   │          0 │ mask_decoder[1][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ masked_features     │ (None, 1692, 64)  │          0 │ identity_encoder… │
│ (MaskedFeatures)    │                   │            │ qmask[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ masked_features_1   │ (None, 1692, 64)  │          0 │ identity_encoder

 Total params: 79,337 (309.91 KB)

 Trainable params: 79,017 (308.66 KB)

 Non-trainable params: 320 (1.25 KB)

Epoch 1/15
261/261 ━━━━━━━━━━━━━━━━━━━━ 111s 268ms/step - global_identity_loss: 0.0094 - global_identity_mean_absolute_error: 0.1322 - hsp_identity_loss: 0.0125 - hsp_identity_mean_absolute_error: 0.1654 - loss: 1.2856 - qmask_loss: 0.6266 - smask_loss: 0.6361 - val_global_identity_loss: 0.0044 - val_global_identity_mean_absolute_error: 0.0820 - val_hsp_identity_loss: 0.0110 - val_hsp_identity_mean_absolute_error: 0.1546 - val_loss: 10.7579 - val_qmask_loss: 5.3819 - val_smask_loss: 5.3605
Epoch 2/15
261/261 ━━━━━━━━━━━━━━━━━━━━ 27s 104ms/step - global_identity_loss: 0.0017 - global_identity_mean_absolute_error: 0.0467 - hsp_identity_loss: 0.0044 - hsp_identity_mean_absolute_error: 0.0746 - loss: 0.6688 - qmask_loss: 0.3270 - smask_loss: 0.3358 - val_global_identity_loss: 0.0017 - val_global_identity_mean_absolute_error: 0.0484 - val_hsp_identity_loss: 0.0029 - val_hsp_identity_mean_absolute_error: 0.0587 - val_loss: 0.6307 - val_qmask_loss: 0.3075 - val_smask_loss: 0.3179
Epoch 3/15
2

In [4]:
import os
import csv
import json
import time
import numpy as np
import tensorflow as tf
from Bio import SeqIO
from tqdm import tqdm
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde, friedmanchisquare, wilcoxon
from statsmodels.stats.multitest import multipletests
import pandas as pd
import seaborn as sns


def one_hot_encode(seq, max_len):
    mapping = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    arr = np.zeros((max_len, 4), dtype=np.float32)
    for i, base in enumerate(seq[:max_len]):
        if base in mapping:
            arr[i, mapping[base]] = 1.0
    return arr

def mask_iou(true_mask, pred_mask, threshold=0.5):
    pred_mask_bin = (pred_mask > threshold).astype(np.float32)
    intersection = np.sum(true_mask * pred_mask_bin, axis=1)
    union = np.sum(np.maximum(true_mask, pred_mask_bin), axis=1) + 1e-7
    return np.mean(intersection / union)

def load_test_data(fasta_file, tsv_file, max_len, hmin, hmax, gmin, gmax):
    sequences = {r.id: str(r.seq) for r in SeqIO.parse(fasta_file, "fasta")}

    q_seqs, s_seqs = [], []
    qmask_true, smask_true = [], []
    hsp_true_norm, glob_true_norm = [], []

    with open(tsv_file) as f:
        reader = csv.DictReader(f, delimiter="\t")
        for row in tqdm(reader, desc="Loading test pairs"):
            qid, sid = row["seq_id_1"], row["seq_id_2"]
            if qid not in sequences or sid not in sequences or not row["hsp_identity"].strip():
                continue

            q = one_hot_encode(sequences[qid], max_len)
            s = one_hot_encode(sequences[sid], max_len)

            qm = np.zeros(max_len, dtype=np.float32)
            sm = np.zeros(max_len, dtype=np.float32)
            qstart, qend = int(row["qstart"]), int(row["qend"])
            sstart, send = int(row["sstart"]), int(row["send"])
            qm[qstart:qend + 1] = 1.0
            sm[sstart:send + 1] = 1.0

            hsp_val  = float(row["hsp_identity"])
            glob_val = float(row["global_identity"])
            hsp_norm  = (hsp_val - hmin) / (hmax - hmin)
            glob_norm = (glob_val - gmin) / (gmax - gmin)

            q_seqs.append(q)
            s_seqs.append(s)
            qmask_true.append(qm)
            smask_true.append(sm)
            hsp_true_norm.append(hsp_norm)
            glob_true_norm.append(glob_norm)

    return (
        np.array(q_seqs, dtype=np.float32),
        np.array(s_seqs, dtype=np.float32),
        np.array(qmask_true, dtype=np.float32),
        np.array(smask_true, dtype=np.float32),
        np.array(hsp_true_norm, dtype=np.float32),
        np.array(glob_true_norm, dtype=np.float32),
    )




class AbsDiff(tf.keras.layers.Layer):
    def call(self, inputs):
        a, b = inputs
        return tf.abs(a - b)


class OutputLayer(tf.keras.layers.Layer):
    def __init__(self, name=None, **kwargs):
        super().__init__(name=name, **kwargs)

    def call(self, inputs):
        return inputs


class MaskCoverage(tf.keras.layers.Layer):
    def call(self, mask):
        return tf.reduce_mean(mask, axis=1)


class UniformMask(tf.keras.layers.Layer):
    def __init__(self, seq_len, **kwargs):
        super().__init__(**kwargs)
        self.seq_len = seq_len

    def call(self, x):
        batch_size = tf.shape(x)[0]
        return tf.ones((batch_size, self.seq_len, 1), dtype=tf.float32) / self.seq_len


class MaskedFeatures(tf.keras.layers.Layer):
    def call(self, inputs):
        features, mask = inputs

        if len(mask.shape) == 2:
            mask = tf.expand_dims(mask, -1)

        return features * mask


class CrossAttention(tf.keras.layers.Layer):
    def call(self, inputs):
        q_enc, s_enc = inputs
        scores = tf.matmul(q_enc, s_enc, transpose_b=True)
        probs = tf.nn.softmax(scores, axis=-1)
        return tf.matmul(probs, s_enc)


class MaskWeightedPooling(tf.keras.layers.Layer):
    def call(self, inputs):
        features, mask = inputs
        mask = tf.cast(mask, tf.float32)
        if len(mask.shape) == 2:
            mask = tf.expand_dims(mask, -1)

        numerator = tf.reduce_sum(features * mask, axis=1)
        denominator = tf.reduce_sum(mask, axis=1) + 1e-6
        return numerator / denominator


class ZeroLike(tf.keras.layers.Layer):
    def call(self, x):
        return tf.zeros_like(x)


class IdentityProjection(tf.keras.layers.Layer):
    def __init__(self, feature_dim, **kwargs):
        super().__init__(**kwargs)
        self.feature_dim = feature_dim

    def call(self, identity):
        return tf.repeat(identity, repeats=self.feature_dim, axis=-1)

    def get_config(self):
        config = super().get_config()
        config.update({"feature_dim": self.feature_dim})
        return config


class ResidualScaling(tf.keras.layers.Layer):
    def __init__(self, scale=0.1, **kwargs):
        super().__init__(**kwargs)
        self.scale = scale

    def call(self, x):
        return self.scale * x

    def get_config(self):
        config = super().get_config()
        config.update({"scale": self.scale})
        return config


class MaskedAverage(tf.keras.layers.Layer):
    def call(self, inputs):
        features, mask = inputs

        if len(mask.shape) == 2:
            mask = tf.expand_dims(mask, -1)

        numerator = tf.reduce_sum(features * mask, axis=1)
        denominator = tf.reduce_sum(mask, axis=1) + 1e-6

        return numerator / denominator


class StructuralModelF(tf.keras.layers.Layer):
    def build(self, input_shape):
        self.b1 = self.add_weight(shape=(1,), initializer="ones")
        self.b2 = self.add_weight(shape=(1,), initializer="zeros")
        self.b3 = self.add_weight(shape=(1,), initializer="zeros")
        self.b4 = self.add_weight(shape=(1,), initializer="zeros")
        self.b5 = self.add_weight(shape=(1,), initializer="zeros")
        self.bias = self.add_weight(shape=(1,), initializer="zeros")

    def call(self, inputs):
        hsp_identity, coverage = inputs

        return (
                self.b1 * hsp_identity +
                self.b2 * coverage +
                self.b3 * (hsp_identity * coverage) +
                self.b4 * tf.square(hsp_identity) +
                self.b5 * tf.square(coverage) +
                self.bias
        )




def scatter(true, pred, label, save_path):
    plt.figure(figsize=(4,3), dpi=200)
    xy = np.vstack([true, pred])
    z = gaussian_kde(xy)(xy)
    idx = z.argsort()
    true, pred, z = true[idx], pred[idx], z[idx]
    sc = plt.scatter(true, pred, c=z, s=8, cmap="viridis")
    lims = [min(true.min(), pred.min()), max(true.max(), pred.max())]
    plt.plot(lims, lims, linestyle="--", color="orange", linewidth=1.2)

    plt.xlabel("True", fontsize=10)
    plt.ylabel("Predicted", fontsize=10)
    plt.title(label, fontsize=10)
    plt.xticks(fontsize=10)
    plt.yticks(fontsize=10)
    plt.grid(True, linewidth=0.4)

    cbar = plt.colorbar(sc)
    cbar.set_label("Density", fontsize=10)
    cbar.ax.tick_params(labelsize=10)

    plt.tight_layout()
    plt.savefig(save_path, bbox_inches="tight", pad_inches=0.05)
    plt.close()



DATASETS_ROOT = "/content/drive/MyDrive/seq_similarity_8-7/datasets_1-12"

dataset_dirs = sorted([
    os.path.join(DATASETS_ROOT, d)
    for d in os.listdir(DATASETS_ROOT)
    if d.startswith("dataset_")
])

all_results_rows = []
all_agg_rows = []
all_quantile_rows = []

for data_dir in dataset_dirs:

    print(f"\n==============================")
    print(f"DATASET: {data_dir}")
    print(f"==============================")

    ablation_name = "fixed_model"
    ablation_root = os.path.join(data_dir, "fixed_model_runs")

    if not os.path.exists(ablation_root):
        print("No trained models found, skipping...")
        continue

    metadata_path = os.path.join(data_dir, "metadata.json")
    test_fasta = os.path.join(data_dir, "test_sequences.fasta")
    test_tsv = os.path.join(data_dir, "test_labels.tsv")

    results_path = os.path.join(ablation_root, "test_results.tsv")

    with open(metadata_path) as f:
        metadata = json.load(f)

    MAX_SEQ_LEN = metadata["max_sequence_length"]
    hmin, hmax = metadata["hsp_identity"]["min"], metadata["hsp_identity"]["max"]
    gmin, gmax = metadata["global_identity"]["min"], metadata["global_identity"]["max"]

    q_test, s_test, qmask_true, smask_true, hsp_true, glob_true = load_test_data(test_fasta, test_tsv, MAX_SEQ_LEN, hmin, hmax, gmin, gmax)

    dist_dir = os.path.join(ablation_root, "test_distributions")
    os.makedirs(dist_dir, exist_ok=True)

    hsp_true_orig  = hsp_true * (hmax - hmin) + hmin
    glob_true_orig = glob_true * (gmax - gmin) + gmin

    plt.figure(figsize=(4,3), dpi=200)
    plt.hist(hsp_true_orig, bins=100, density=False, color="#450558")
    plt.xlabel("HSP Identity", fontsize=10)
    plt.ylabel("Count", fontsize=10)
    plt.title("Test Distribution: HSP Identity", fontsize=10)
    plt.grid(True, linewidth=0.4)
    plt.tight_layout()
    plt.savefig(os.path.join(dist_dir, "hsp_identity_distribution.png"), bbox_inches="tight", pad_inches=0.05)
    plt.close()

    plt.figure(figsize=(4,3), dpi=200)
    plt.hist(glob_true_orig, bins=100, density=False, color="#450558")
    plt.xlabel("Global Identity", fontsize=10)
    plt.ylabel("Count", fontsize=10)
    plt.title("Test Distribution: Global Identity", fontsize=10)
    plt.grid(True, linewidth=0.4)
    plt.tight_layout()
    plt.savefig(os.path.join(dist_dir, "global_identity_distribution.png"), bbox_inches="tight", pad_inches=0.05)
    plt.close()

    print("\nSaved test distribution histograms.")


    results_rows = []
    quantile_rows = []


    for run_name in sorted(os.listdir(ablation_root)):

        run_dir = os.path.join(ablation_root, run_name)
        if not os.path.isdir(run_dir):
            continue

        model_path = os.path.join(run_dir, "model.keras")
        if not os.path.exists(model_path):
            continue

        print(f"\n--- {run_name} ---")

        tf.keras.backend.clear_session()

        model = tf.keras.models.load_model(
            model_path,
            custom_objects={
                "AbsDiff": AbsDiff,
                "IdentityProjection": IdentityProjection,
                "MaskedAverage": MaskedAverage,
                "MaskedFeatures": MaskedFeatures,
                "OutputLayer": OutputLayer,
                "CrossAttention": CrossAttention,
                "MaskWeightedPooling": MaskWeightedPooling,
                "ResidualScaling": ResidualScaling,
                "MaskCoverage": MaskCoverage,
                "UniformMask": UniformMask,
                "ZeroLike": ZeroLike,
                "StructuralModelF": StructuralModelF
            },
            compile=False
        )

        start_time = time.process_time()
        preds = model.predict({"q_seq": q_test, "s_seq": s_test}, batch_size=32, verbose=1)
        cpu_time = time.process_time() - start_time
        avg_time_per_pair = cpu_time / len(q_test)

        hsp_pred = preds["hsp_identity"].squeeze()
        glob_pred = preds["global_identity"].squeeze()
        hsp_true_orig  = hsp_true * (hmax - hmin) + hmin
        hsp_pred_orig  = hsp_pred * (hmax - hmin) + hmin
        glob_true_orig = glob_true * (gmax - gmin) + gmin
        glob_pred_orig = glob_pred * (gmax - gmin) + gmin

        qmask_pred = preds["qmask"].squeeze()
        smask_pred = preds["smask"].squeeze()
        iou_q = mask_iou(qmask_true, qmask_pred)
        iou_s = mask_iou(smask_true, smask_pred)

        print("\n===== Mask Evaluation (IoU) =====")
        print(f"Query mask IoU:", round(iou_q, 3))
        print(f"Subject mask IoU:", round(iou_s, 3))

        print("\n===== HSP Identity =====")
        mae_hsp = mean_absolute_error(hsp_true_orig, hsp_pred_orig)
        rmse_hsp = np.sqrt(mean_squared_error(hsp_true_orig, hsp_pred_orig))
        print("MAE :", round(mae_hsp, 3))
        print("RMSE:", round(rmse_hsp, 3))

        print("\n===== Global Identity =====")
        mae_glob = mean_absolute_error(glob_true_orig, glob_pred_orig)
        rmse_glob = np.sqrt(mean_squared_error(glob_true_orig, glob_pred_orig))
        pearson_r = np.corrcoef(glob_true_orig, glob_pred_orig)[0, 1]
        r2 = r2_score(glob_true_orig, glob_pred_orig)
        mre = np.mean(np.abs(glob_true_orig - glob_pred_orig) / np.maximum(1e-6, np.abs(glob_true_orig)))
        print("MAE :", round(mae_glob, 3))
        print("RMSE:", round(rmse_glob, 3))
        print("Pearson r:", round(pearson_r, 3))
        print("R²:", round(r2, 3))
        print("MRE :", round(mre, 3))
        print("MRE %:", round(mre*100, 3))

        print("\n===== Inference Timing =====")
        print(f"Total CPU time: {round(cpu_time, 3)} sec")
        print(f"Avg inference time per pair: {round(avg_time_per_pair * 1000, 3)} ms")

        row = [
            ablation_name, run_name,
            round(iou_q, 4), round(iou_s, 4),
            round(mae_hsp, 4), round(rmse_hsp, 4),
            round(mae_glob, 4), round(rmse_glob, 4),
            round(pearson_r, 4), round(r2, 4),
            round(mre, 4),
            round(avg_time_per_pair * 1000, 4)
        ]

        results_rows.append(row)
        all_results_rows.append([os.path.basename(data_dir)] + row)


        scatter(hsp_true_orig, hsp_pred_orig, "HSP identity", os.path.join(run_dir, "test_scatter_hsp_identity.png"))
        scatter(glob_true_orig, glob_pred_orig, "Global identity", os.path.join(run_dir, "test_scatter_global_identity.png"))

        num_bins = 10
        print("\n===== HSP MAE per Quantile =====")
        quantiles = np.quantile(hsp_true_orig, np.linspace(0, 1, num_bins + 1))
        quantiles = np.unique(quantiles)
        bin_ids = np.digitize(hsp_true_orig, quantiles[1:-1], right=True)

        for i in range(num_bins):
            mask = bin_ids == i
            if np.sum(mask) > 0:
                mae_bin = mean_absolute_error(hsp_true_orig[mask], hsp_pred_orig[mask])
                print(f"Bin {i}: MAE = {round(mae_bin, 3)}")

                qrow = [ablation_name, run_name, f"hsp_quantile_{i}", round(mae_bin, 4)]
                quantile_rows.append(qrow)
                all_quantile_rows.append([os.path.basename(data_dir)] + qrow)


        print("\n===== Global MAE per Quantile =====")
        quantiles_g = np.quantile(glob_true_orig, np.linspace(0, 1, num_bins + 1))
        quantiles_g = np.unique(quantiles_g)
        bin_ids_g = np.digitize(glob_true_orig, quantiles_g[1:-1], right=True)

        for i in range(num_bins):
            mask_g = bin_ids_g == i
            if np.sum(mask_g) > 0:
                mae_bin_g = mean_absolute_error(glob_true_orig[mask_g], glob_pred_orig[mask_g])
                print(f"Bin {i}: MAE = {round(mae_bin_g, 3)}")

                qrow = [ablation_name, run_name, f"global_quantile_{i}", round(mae_bin_g, 4)]
                quantile_rows.append(qrow)
                all_quantile_rows.append([os.path.basename(data_dir)] + qrow)

    with open(results_path, "w", newline="") as f:
        writer = csv.writer(f, delimiter="\t")
        writer.writerow([
            "ablation","run","iou_q","iou_s",
            "mae_hsp","rmse_hsp","mae_global","rmse_global",
            "pearson_r","r2","mre","avg_ms_per_pair"
        ])
        writer.writerows(results_rows)

    df = pd.DataFrame(results_rows, columns=[
        "ablation","run","iou_q","iou_s",
        "mae_hsp","rmse_hsp","mae_global","rmse_global",
        "pearson_r","r2","mre","avg_ms_per_pair"
    ])


    metrics = [
        "iou_q",
        "iou_s",
        "mae_hsp",
        "rmse_hsp",
        "mae_global",
        "rmse_global",
        "pearson_r",
        "r2",
        "mre"
    ]

    agg_rows = []

    for ablation in df["ablation"].unique():
        subset = df[df["ablation"] == ablation]
        row = {"ablation": ablation}
        for metric in metrics:
            mean_val = subset[metric].mean()
            std_val = subset[metric].std()
            row[metric] = f"{mean_val:.4f} ± {std_val:.4f}"
        agg_rows.append(row)

    agg_df = pd.DataFrame(agg_rows)
    agg_df.to_csv(results_path.replace(".tsv","_aggregated.tsv"), sep="\t", index=False)

    pd.DataFrame(
        quantile_rows,
        columns=["ablation","run","quantile_bin","mae"]
    ).to_csv(results_path.replace(".tsv","_quantiles.tsv"), sep="\t", index=False)




all_results_df = pd.DataFrame(
    all_results_rows,
    columns=["dataset","ablation","run","iou_q","iou_s",
             "mae_hsp","rmse_hsp","mae_global","rmse_global",
             "pearson_r","r2","mre","avg_ms_per_pair"]
)

all_results_df.to_csv(
    os.path.join(DATASETS_ROOT, "ALL_DATASETS_RESULTS.tsv"),
    sep="\t", index=False
)

all_quantile_df = pd.DataFrame(
    all_quantile_rows,
    columns=["dataset","ablation","run","quantile_bin","mae"]
)

all_quantile_df.to_csv(
    os.path.join(DATASETS_ROOT, "ALL_DATASETS_QUANTILES.tsv"),
    sep="\t", index=False
)



summary_df = all_results_df.groupby("dataset").agg({
    "iou_q": ["mean","std"],
    "iou_s": ["mean","std"],
    "mae_hsp": ["mean","std"],
    "rmse_hsp": ["mean","std"],
    "mae_global": ["mean","std"],
    "rmse_global": ["mean","std"],
    "pearson_r": ["mean","std"],
    "r2": ["mean","std"],
    "mre": ["mean","std"]
})

summary_df.columns = ['_'.join(col) for col in summary_df.columns]
summary_df.reset_index(inplace=True)

formatted_rows = []

for _, row in summary_df.iterrows():
    new_row = {"dataset": row["dataset"]}

    for metric in [
        "iou_q", "iou_s",
        "mae_hsp", "rmse_hsp",
        "mae_global", "rmse_global",
        "pearson_r", "r2", "mre"
    ]:
        mean_val = row[f"{metric}_mean"]
        std_val  = row[f"{metric}_std"]

        if pd.isna(std_val):
            std_val = 0.0

        new_row[metric] = f"{mean_val:.4f} ± {std_val:.4f}"

    formatted_rows.append(new_row)

formatted_summary_df = pd.DataFrame(formatted_rows)

formatted_summary_df.to_csv(
    os.path.join(DATASETS_ROOT, "ALL_DATASETS_SUMMARY.tsv"),
    sep="\t",
    index=False
)




DATASET: /content/drive/MyDrive/seq_similarity_8-7/datasets_1-12/dataset_10


Loading test pairs: 11896it [00:08, 1391.67it/s]



Saved test distribution histograms.

--- run_0 ---
372/372 ━━━━━━━━━━━━━━━━━━━━ 29s 46ms/step

===== Mask Evaluation (IoU) =====
Query mask IoU: 0.937
Subject mask IoU: 0.935

===== HSP Identity =====
MAE : 0.598
RMSE: 0.939

===== Global Identity =====
MAE : 1.296
RMSE: 1.761
Pearson r: 0.976
R²: 0.953
MRE : 0.017
MRE %: 1.735

===== Inference Timing =====
Total CPU time: 45.901 sec
Avg inference time per pair: 3.859 ms

===== HSP MAE per Quantile =====
Bin 0: MAE = 0.574
Bin 1: MAE = 0.63
Bin 2: MAE = 0.536
Bin 3: MAE = 0.394
Bin 4: MAE = 0.468
Bin 5: MAE = 0.512
Bin 6: MAE = 0.706
Bin 7: MAE = 0.703
Bin 8: MAE = 0.834
Bin 9: MAE = 0.619

===== Global MAE per Quantile =====
Bin 0: MAE = 2.003
Bin 1: MAE = 1.217
Bin 2: MAE = 1.114
Bin 3: MAE = 0.961
Bin 4: MAE = 0.972
Bin 5: MAE = 1.18
Bin 6: MAE = 1.501
Bin 7: MAE = 1.326
Bin 8: MAE = 1.48
Bin 9: MAE = 1.202
